<a href="https://colab.research.google.com/github/areslou/cpecog2-coral-bleaching/blob/main/notebooks/Group2_Coral_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Coral bleaching demo - Run 10

This notebook launches the coral bleaching classification app in Google Colab.

- Model: Run 10, three YOLOv8s-cls students (seeds 42, 43, 44) trained with knowledge distillation from DINOv2; their P(bleached) values are averaged
- Files: `results/run10/final_model.json` (lists the three models and the cutoff) and `results/run10/final_model_s42.pt`, `_s43.pt`, `_s44.pt`
- Bleaching cutoff: `0.127` (chosen from 5-fold cross-validation, bleached recall of at least 0.90)
- Grad-CAM heatmaps come from the seed-42 model
- The model is a screening aid; a trained reef monitor makes the final decision.

For a browser demo, use a GPU runtime and run every cell. Keep the final cell running while using the `gradio.live` link.

In [ ]:
!if [ -d cpecog2-coral-bleaching/.git ]; then cd cpecog2-coral-bleaching; else git clone https://github.com/areslou/cpecog2-coral-bleaching.git && cd cpecog2-coral-bleaching; fi

In [ ]:
!pip install -q ultralytics==8.4.211 pi-heif gradio==6.32.0 torch torchvision kagglehub pandas pillow numpy

In [ ]:
import os
import torch

os.environ.setdefault("GRADIO_ANALYTICS_ENABLED", "False")
os.environ.setdefault("YOLO_AUTOINSTALL", "false")
os.environ.setdefault("YOLO_OFFLINE", "true")

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("PyTorch:", torch.__version__)

## 1. Check the Run 10 model

The model files should be downloaded as part of the repository. `final_model.json` lists the three seed models, how they are combined, and the cutoff.

In [ ]:
WEIGHTS = "results/run10/final_model.json"
THRESHOLD = None   # None uses the cross-validated cutoff saved in final_model.json (0.127)

import json
from demo import model_files, model_size_mb, default_threshold

if not os.path.exists(WEIGHTS):
    raise FileNotFoundError(
        f"{WEIGHTS} was not found. Run this notebook from the repository with the Run 10 files, "
        "or update WEIGHTS to the downloaded final_model.json (or a single .pt file)."
    )
missing = [f for f in model_files(WEIGHTS) if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f"Missing model files listed in {WEIGHTS}: {missing}")
print(json.dumps(json.load(open(WEIGHTS, encoding="utf-8")), indent=2))
print("Model files:", [os.path.basename(f) for f in model_files(WEIGHTS)])
print("Threshold:", THRESHOLD if THRESHOLD is not None else default_threshold(WEIGHTS)[0])
print("Size MB:", round(model_size_mb(WEIGHTS), 2))

## 2. Optional: create demo photographs

This downloads the held-out test photographs once. Skip this cell if you will upload your own reef photographs.

In [ ]:
#!python get_demo_photos.py
#!python get_demo_photos.py --sites

## 3. Launch the demo app

The public link is created with Gradio `share=True`. Upload either:

- one reef photograph in the **Single photo** tab, or
- photographs and/or a `photo,site` CSV in the **Survey summary** tab.

In [ ]:
from app import build

ui = build(WEIGHTS, THRESHOLD)
ui.launch(server_name="0.0.0.0", server_port=7860, share=True)